# 🎬 ProXMax — স্বয়ংক্রিয় English → Bangla ভিডিও ডাবিং

**Google Drive**-এর `/content/drive/MyDrive/Video/input` ফোল্ডারে থাকা সব ইংরেজি ভিডিও স্বয়ংক্রিয়ভাবে বাংলায় ডাব করে `/content/drive/MyDrive/Video/output` ফোল্ডারে সেভ করবে।

| ধাপ | প্রযুক্তি |
|---|---|
| ১. ইংরেজি কথা শনাক্তকরণ (STT) | `faster-whisper` (`large-v3` প্রি-লোডেড + FC Mobile ও ফুটবল `hotwords.txt` + সীমান্ত মেরামত) |
| ২. ইংরেজি → বাংলা অনুবাদ | **Google Gemini API** (ডিফল্ট `gemini-3.5-flash-lite`, স্লট-বাজেট ও হটওয়ার্ড সচেতন) |
| ৩. বাংলা ভয়েস তৈরি (TTS) | **বিল্ট-ইন বাংলা neural ভয়েস** (`edge-tts` — পুরুষ কণ্ঠ `bn-BD-PradeepNeural` ডিফল্ট, নারী `bn-BD-NabanitaNeural`) |
| ৪. অডিও-ভিডিও সিঙ্ক | `ffmpeg` (`rubberband` / `atempo`, শূন্য ওভারল্যাপ গ্যারান্টি) |

---
### ব্যবহারবিধি
নিচের **৫টি ধাপ** উপর থেকে নিচে একে একে `Shift + Enter` চেপে চালান।

## ধাপ ১: Google Drive মাউন্ট

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ধাপ ২: রিপোজিটরি ক্লোন ও ফোল্ডার প্রস্তুত

In [ ]:
import os, pathlib, subprocess

REPO_URL = "https://github.com/redwan-primeAmax/ProXMax.git"
REPO_DIR = "/content/ProXMax"

if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)

# ইনপুট ও আউটপুট ফোল্ডার নিশ্চিত করা
pathlib.Path("/content/drive/MyDrive/Video/input").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/Video/output").mkdir(parents=True, exist_ok=True)
print("✅ বর্তমান ডিরেক্টরি:", os.getcwd())

## ধাপ ৩: দ্রুত ডিপেন্ডেন্সি ইনস্টল ও Whisper `large-v3` মডেল প্রি-ডাউনলোড
এই ধাপেই `faster-whisper` (`large-v3`) মডেলটি ক্যাশে ডাউনলোড হয়ে যাবে, তাই ভিডিও প্রসেসিংয়ের সময় আর নতুন করে ডাউনলোড হবে না।

In [ ]:
import os, shutil, subprocess, sys, time

t0 = time.perf_counter()

# ১. সিস্টেম টুল (ffmpeg / rubberband) কেবল না থাকলেই ইনস্টল হবে
missing_apt = []
if not shutil.which("ffmpeg"):
    missing_apt.append("ffmpeg")
if not shutil.which("rubberband"):
    missing_apt.append("rubberband-cli")
if missing_apt:
    print(f"📦 সিস্টেম টুল ইনস্টল হচ্ছে: {', '.join(missing_apt)}...")
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "--no-install-recommends", *missing_apt],
        check=False,
    )

# ২. পাইথন প্যাকেজ দ্রুত ইনস্টল (Colab-এ আগে থাকা প্যাকেজ স্কিপ হবে)
print("⚡ পাইথন ডিপেন্ডেন্সি ইনস্টল হচ্ছে...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "-r", "requirements.txt",
    ],
    check=True,
)

# ৩. Whisper large-v3 মডেল এখনই ডাউনলোড ও ক্যাশ করা (যাতে ডাবিংয়ের সময় ডাউনলোড না হয়)
if "/content/ProXMax" not in sys.path:
    sys.path.insert(0, "/content/ProXMax")

from proxmax.stt import ensure_whisper_model
from proxmax.hotwords import load_hotwords

WHISPER_MODEL = "large-v3"  # @param ["large-v3", "medium", "distil-large-v3"]
print(f"📥 Whisper '{WHISPER_MODEL}' মডেল ক্যাশে প্রস্তুত করা হচ্ছে...")
cached_model_dir = ensure_whisper_model(WHISPER_MODEL)
hotword_count = len(load_hotwords("hotwords.txt"))

print(f"✅ সেটআপ সম্পন্ন ({time.perf_counter() - t0:.1f}s)!")
print(f"   • Whisper মডেল ক্যাশ: {cached_model_dir}")
print(f"   • লোড হওয়া Hotwords (FC Mobile + Players + Events): {hotword_count} টি")

## ধাপ ৪: Gemini API ও মডেল/ইনপুট বাজেট সেট করুন (লাইভ API যাচাই সহ)
১. API key নিন: **https://aistudio.google.com/app/apikey**।
২. নিচের API key বক্সে পেস্ট করুন (অথবা Colab Secrets-এ `GEMINI_API_KEY` রাখুন)।
৩. মডেল বদলান এবং প্রতি request-এর input token cap দিন: default **5000**; চাইলে 2000, 10000, 20000 বা যেকোনো custom সংখ্যা (10000+) লিখতে পারবেন; সর্বনিম্ন 1000।
৪. সেল চালালে API key ও মডেল test হবে। Token cap প্রতি Gemini request-এর জন্য; মডেলের নিজস্ব context limit-এর বেশি পাঠানো যাবে না।


In [ ]:
import os, sys
if "/content/ProXMax" not in sys.path:
    sys.path.insert(0, "/content/ProXMax")

# @markdown ### 🔑 এখানে আপনার Gemini API Key দিন:
GEMINI_API_KEY = ""  # @param {type:"string"}
# @markdown ### 🤖 অনুবাদের মডেল:
GEMINI_MODEL = "gemini-3.5-flash-lite"  # @param ["gemini-3.5-flash-lite", "gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.0-flash"] {allow-input: true}
# @markdown ### 🧮 প্রতি request-এর সর্বোচ্চ input token (ডিফল্ট 5000; 10000+ custom মানও লেখা যাবে):
GEMINI_MAX_INPUT_TOKENS = 5000  # @param {type:"integer"}
# @markdown ### 🗣️ বাংলা কণ্ঠ বেছে নিন:
# @markdown - `male` = পুরুষ কণ্ঠ (`bn-BD-PradeepNeural`, ডিফল্ট)
# @markdown - `female` = নারী কণ্ঠ (`bn-BD-NabanitaNeural`)
TTS_GENDER = "male"  # @param ["male", "female"]

if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") or ""
    except Exception:
        pass

GEMINI_API_KEY = GEMINI_API_KEY.strip()
GEMINI_MAX_INPUT_TOKENS = int(GEMINI_MAX_INPUT_TOKENS)
if GEMINI_MAX_INPUT_TOKENS < 1000:
    raise ValueError("GEMINI_MAX_INPUT_TOKENS অন্তত 1000 হতে হবে।")
if not GEMINI_API_KEY:
    raise ValueError(
        "❌ GEMINI_API_KEY খালি! উপরের বক্সে আপনার Gemini API key পেস্ট করে সেলটি আবার চালান।"
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["PROXMAX_GEMINI_MODEL"] = GEMINI_MODEL.strip() or "gemini-3.5-flash-lite"
os.environ["PROXMAX_GEMINI_MAX_INPUT_TOKENS"] = str(GEMINI_MAX_INPUT_TOKENS)
os.environ["PROXMAX_TTS_GENDER"] = TTS_GENDER.strip() or "male"

# API Key সত্যিই কাজ করছে কি না তা এখনই টেস্ট করা হচ্ছে
from proxmax.translator import verify_gemini_api

print("🔍 Gemini API Key যাচাই করা হচ্ছে...")
check = verify_gemini_api(
    api_key=GEMINI_API_KEY,
    model_name=os.environ["PROXMAX_GEMINI_MODEL"],
)

# ফলো-আপ মডেল পরিবর্তন হলে তা পরিবেশে আপডেট রাখা
os.environ["PROXMAX_GEMINI_MODEL"] = check["active_model"]
voice_label = "পুরুষ (Pradeep)" if TTS_GENDER == "male" else "নারী (Nabanita)"
print(f"✅ Gemini API Key সচল এবং কাজ করছে!")
print(f"   • সক্রিয় মডেল : {check['active_model']}")
print(f"   • টেস্ট অনুবাদ : {check['sample_reply']}")
print(f"   • বাংলা কণ্ঠ   : {voice_label}")

## ধাপ ৫: সব ভিডিও ডাব করুন!
নিচের সেলটি চালালেই `/content/drive/MyDrive/Video/input` ফোল্ডারের সব ভিডিও একে একে বাংলায় ডাব হয়ে `/content/drive/MyDrive/Video/output` ফোল্ডারে সেভ হবে। প্রতিটি ভিডিওর জন্য বিস্তারিত transcript ও sync diagnostics-সহ `<video_stem>_report.markdown` রিপোর্ট তৈরি হবে; আলাদা `_en.srt` / `_bn.srt` ফাইল তৈরি হবে না। রিপোর্টের action timeline কথোপকথন থেকে তৈরি—এটি on-screen visual action recognition নয়।

In [ ]:
import os
from proxmax.config import AppConfig
from proxmax.pipeline import run_Single_or_batch
from proxmax.utils import setup_logging

config = AppConfig.from_yaml("config.yaml")

# উপরের সেলগুলোর সেটিংস প্রয়োগ
if globals().get("WHISPER_MODEL"):
    config.whisper_model = WHISPER_MODEL
if os.environ.get("PROXMAX_GEMINI_MODEL"):
    config.gemini_model = os.environ["PROXMAX_GEMINI_MODEL"]
if os.environ.get("PROXMAX_GEMINI_MAX_INPUT_TOKENS"):
    config.gemini_max_input_tokens = int(os.environ["PROXMAX_GEMINI_MAX_INPUT_TOKENS"])
if os.environ.get("PROXMAX_TTS_GENDER"):
    config.tts_gender = os.environ["PROXMAX_TTS_GENDER"]

# চাইলে এখানে যেকোনো সেটিং কাস্টমাইজ করতে পারেন:
# config.input_dir = "/content/drive/MyDrive/Video/input"
# config.output_dir = "/content/drive/MyDrive/Video/output"
# config.stt_hotwords = "CustomName1, CustomName2"  # hotwords.txt-এর পাশাপাশি অতিরিক্ত শব্দ

setup_logging(config.log_level)
summary = run_Single_or_batch(config)

print(f"\n🎉 সফল: {len(summary.succeeded)} | স্কিপ: {len(summary.skipped)} | ব্যর্থ: {len(summary.failed)}")